In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from nemo_platform import NeMoPlatform
from nemo_platform.beta.safe_synthesizer.job_builder import SafeSynthesizerJobBuilder

client = NeMoPlatform(
    base_url="http://localhost:8080",
    inference_base_url="http://localhost:8080",
)
# leave commented out - we inject this path for testing in CI
# csv_filepath = "womens-clothing-ecommerce.csv"
data = pd.read_csv(csv_filepath)  # noqa
builder = (
    SafeSynthesizerJobBuilder(client)
    .with_data_source(data)
    .synthesize()
    .with_generate(use_structured_gen=True)
    .with_replace_pii()
)

In [2]:
job_name = f"synthesis-test-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"
project_name = "test-project"
job = builder.create_job(name=job_name, project=project_name)
job.wait_for_completion()

{"asctime": "2025-12-01 16:03:53,195", "process": 782873, "level": "INFO", "name": "httpx", "message": "HTTP Request: POST http://localhost:8080/v1beta1/safe-synthesizer/jobs \"HTTP/1.1 200 OK\""}
{"asctime": "2025-12-01 16:03:53,227", "process": 782873, "level": "INFO", "name": "httpx", "message": "HTTP Request: GET http://localhost:8080/v1beta1/safe-synthesizer/jobs/job-wrbqtuezvvzuu3b3hdarpx/status \"HTTP/1.1 200 OK\""}
Job status changed to status: 'created', status_details: {}, error_details: None
Job status changed to status: 'pending', status_details: {}, error_details: None
{"asctime": "2025-12-01 16:04:06,119", "process": 1, "level": "INFO", "name": "matplotlib.font_manager", "message": "generated new fontManager"}
{"asctime": "2025-12-01 16:04:08,619", "process": 1, "level": "INFO", "name": "faiss.loader", "message": "Loading faiss with AVX2 support."}
{"asctime": "2025-12-01 16:04:08,706", "process": 1, "level": "INFO", "name": "faiss.loader", "message": "Successfully loaded

In [3]:
current_status = job.fetch_status()
expected_status = "completed"

assert current_status == expected_status, f"Job status mismatch. Expected: '{expected_status}', Got: '{current_status}'"

In [4]:
job.display_report_in_notebook()

In [5]:
current_shape = job.fetch_data().shape
expected_shape = (1000, 11)

assert expected_shape == current_shape, f"Result shape mismatch. Expected: '{expected_shape}', Got: '{current_shape}'"

In [ ]:
# TODO: enable once filter is working
# jobs_response = client.safe_synthesizer.jobs.list(
#    workspace="default",
#    extra_query={"filter[name]": ["synthesis"]}
# )
# print(f"Found {len(jobs_response.data)} jobs matching 'synthesis'")
# assert len(jobs_response.data) >= 1, "Should find at least our job"
#
# matching_job = next((j for j in jobs_response.data if j.name == job_name), None)
# assert matching_job is not None, f"Should find our job {job_name}"
# print(f"✓ Found job by name filter: {matching_job.name}")

In [ ]:
# project_jobs = client.safe_synthesizer.jobs.list(workspace="default", extra_query={"filter[project]": ["test-project"]})
# print(f"Found {len(project_jobs.data)} jobs in project matching 'test-project'")
# assert len(project_jobs.data) >= 1, "Should find at least our job"
#
# matching_project_job = next((j for j in project_jobs.data if j.project == project_name), None)
# assert matching_project_job is not None, f"Should find our job in project {project_name}"
# print(f"✓ Found job by project filter: {matching_project_job.name} in {matching_project_job.project}")

In [ ]:
# combined_filter = client.safe_synthesizer.jobs.list(
#    workspace="default", extra_query={"filter[name]": ["synthesis"], "filter[project]": ["test"]}
# )
# print(f"Found {len(combined_filter.data)} jobs matching both 'synthesis' AND 'test'")
# assert len(combined_filter.data) >= 1, "Should find our job with combined filter"
#
# matching_combined = next((j for j in combined_filter.data if j.id == job.job_id), None)
# assert matching_combined is not None, "Should find our job with combined filter"
# print(f"✓ Found job by combined filter: {matching_combined.name}")